# Fracture Classification — Thesis Analysis Notebook


**Prerequisites:** Run `python train.py` before executing this notebook.

| Section | Content |
|---|---|
| 0 | Setup & load model |
| 1 | Training history (loss & AUC curves) |
| 2 | Core metrics with 95% bootstrap CI — ROC, PR curve, confusion matrix |
| 3 | Per-region subgroup analysis (distal / complete / proximal) |
| 4 | Threshold analysis & clinical operating points |
| 5 | Calibration analysis (reliability diagram, ECE) |
| 6 | Feature space visualisation (t-SNE, UMAP) |
| 7 | Grad-CAM interpretability (TP, TN, FP, FN examples) |
| 8 | Confidence & error analysis |
| 9 | Publication-ready summary table (LaTeX) |

In [ ]:
# ── 0. Setup ───────────────────────────────────────────────────────────────
import sys, warnings
from pathlib import Path

PROJECT_ROOT = Path("../")   # notebook lives in notebooks/, code in project root
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import torch
from PIL import Image

from config import DEVICE, SPLITS_CSV, CHECKPOINT_DIR, LOG_DIR
from dataloader import get_dataloaders, get_val_transforms
from model import build_model
from evaluate import run_inference, compute_metrics, find_optimal_threshold

# ── Figure output directory ────────────────────────────────────────────────
FIG_DIR = LOG_DIR / "thesis_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Plot style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
COLORS = {"fracture": "#e05c5c", "normal": "#4c9be8",
          "train": "#4c9be8", "val": "#e05c5c",
          "correct": "#6dbf8f", "wrong": "#e05c5c"}

print(f"Device  : {DEVICE}")
print(f"FIG_DIR : {FIG_DIR}")

# ── Check training artefacts ───────────────────────────────────────────────
BEST_CKPT  = CHECKPOINT_DIR / "best_model.pth"
TRAIN_LOG  = LOG_DIR / "train_log.csv"
TRAINED    = BEST_CKPT.exists()

if not TRAINED:
    print("\n⚠  best_model.pth not found.")
    print("   Sections 1-9 require a trained model.")
    print("   Run: python train.py")
else:
    print(f"\n✓  Checkpoint found: {BEST_CKPT}")

In [ ]:
# ── Load model & run inference once (cached for all sections) ─────────────
assert TRAINED, "Run python train.py first."

model = build_model(pretrained=False)
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print("Model loaded.")

# ── DataLoaders ───────────────────────────────────────────────────────────
dls = get_dataloaders(SPLITS_CSV)

# ── Inference on test set ─────────────────────────────────────────────────
results = run_inference(model, dls["test"], DEVICE)
labels, probs, preds = results["labels"], results["probs"], results["preds"]
metrics = compute_metrics(results)

# ── Attach region info (from splits.csv) ──────────────────────────────────
splits_df = pd.read_csv(SPLITS_CSV)
test_df   = splits_df[splits_df["split"] == "test"].reset_index(drop=True)
# One row per test image — align with inference results
assert len(test_df) == len(labels), "Mismatch between splits.csv test rows and inference results"
test_df["prob"]  = probs
test_df["pred"]  = preds
test_df["correct"] = (test_df["pred"] == test_df["label_id"]).astype(int)

print(f"Test set: {len(test_df)} images | fracture={labels.sum()} normal={(labels==0).sum()}")

## 1. Training History

In [ ]:
assert TRAIN_LOG.exists(), f"train_log.csv not found at {TRAIN_LOG}"
log = pd.read_csv(TRAIN_LOG)
log["global_epoch"] = range(1, len(log) + 1)

# Phase boundary = last epoch of Phase1
phase1_end = log[log["phase"] == "Phase1"]["global_epoch"].max()
best_epoch = log.loc[log["val_auc"].idxmax(), "global_epoch"]

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

for ax, (tr_col, vl_col, ylabel, title) in zip(axes, [
    ("train_loss", "val_loss", "Loss",    "Training & Validation Loss"),
    ("train_auc",  "val_auc",  "AUC-ROC", "Training & Validation AUC-ROC"),
    (None,         None,       "AUC gap", "Generalisation Gap (train − val AUC)"),
]):
    ax.axvspan(phase1_end + 0.5, log["global_epoch"].max() + 0.5,
               alpha=0.06, color="orange", label="Phase 2")
    ax.axvline(phase1_end + 0.5, color="grey", lw=1, ls="--", label="Phase 1→2")

    if tr_col:  # loss or auc
        ax.plot(log["global_epoch"], log[tr_col], color=COLORS["train"], lw=2, label="Train")
        ax.plot(log["global_epoch"], log[vl_col], color=COLORS["val"],   lw=2, label="Val")
        if vl_col == "val_auc":
            ax.scatter([best_epoch], [log.loc[log["global_epoch"]==best_epoch, vl_col].values[0]],
                       color="gold", zorder=5, s=80, label=f"Best val AUC (ep {best_epoch})")
    else:  # gap
        gap = log["train_auc"] - log["val_auc"]
        ax.fill_between(log["global_epoch"], 0, gap, alpha=0.3, color="purple")
        ax.plot(log["global_epoch"], gap, color="purple", lw=2)
        ax.axhline(0, color="grey", lw=1, ls=":")

    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=8)

fig.suptitle("Training History", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "1_training_history.png", bbox_inches="tight")
plt.show()

## 2. Core Performance Metrics with 95% Bootstrap CI

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.utils import resample

def bootstrap_metrics(labels, probs, n_iter=1000, seed=42):
    """
    Stratified bootstrap (1000 iterations) to compute 95% CI for all metrics.
    Returns dict of metric → (mean, lower_95, upper_95).
    """
    rng = np.random.RandomState(seed)
    records = []
    for _ in range(n_iter):
        idx = resample(np.arange(len(labels)), replace=True,
                       stratify=labels, random_state=rng.randint(0, 99999))
        l, p = labels[idx], probs[idx]
        pred  = (p >= 0.5).astype(int)
        tn = ((pred==0)&(l==0)).sum(); fp = ((pred==1)&(l==0)).sum()
        fn = ((pred==0)&(l==1)).sum(); tp = ((pred==1)&(l==1)).sum()
        records.append({
            "auc":   roc_auc_score(l, p),
            "sens":  tp/(tp+fn) if (tp+fn)>0 else 0,
            "spec":  tn/(tn+fp) if (tn+fp)>0 else 0,
            "acc":   (tp+tn)/len(l),
            "ppv":   tp/(tp+fp) if (tp+fp)>0 else 0,
            "npv":   tn/(tn+fn) if (tn+fn)>0 else 0,
            "f1":    f1_score(l, pred, zero_division=0),
            "ap":    average_precision_score(l, p),
        })
    df = pd.DataFrame(records)
    return {col: (df[col].mean(), df[col].quantile(0.025), df[col].quantile(0.975))
            for col in df.columns}

print("Computing bootstrap CIs (1000 iterations)...")
ci = bootstrap_metrics(labels, probs)

names = {
    "auc":  "AUC-ROC",
    "sens": "Sensitivity",
    "spec": "Specificity",
    "acc":  "Accuracy",
    "ppv":  "PPV (Precision)",
    "npv":  "NPV",
    "f1":   "F1-Score",
    "ap":   "Avg. Precision",
}
rows = []
for k, label in names.items():
    mean, lo, hi = ci[k]
    rows.append({"Metric": label, "Value": f"{mean:.3f}", "95% CI": f"[{lo:.3f}, {hi:.3f}]"})
metrics_table = pd.DataFrame(rows)
print(metrics_table.to_string(index=False))

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

# ── Bootstrap ROC bands ───────────────────────────────────────────────────
from sklearn.utils import resample
rng = np.random.RandomState(42)
mean_fpr = np.linspace(0, 1, 200)
tpr_boot = []
for _ in range(500):
    idx = resample(np.arange(len(labels)), replace=True,
                   stratify=labels, random_state=rng.randint(0,99999))
    fpr_i, tpr_i, _ = roc_curve(labels[idx], probs[idx])
    tpr_boot.append(np.interp(mean_fpr, fpr_i, tpr_i))
tpr_boot = np.array(tpr_boot)
tpr_lo, tpr_hi = np.percentile(tpr_boot, [2.5, 97.5], axis=0)

auc_mean, auc_lo, auc_hi = ci["auc"]
fpr_main, tpr_main, _ = roc_curve(labels, probs)
opt_thresh, opt_sens, opt_spec = find_optimal_threshold(labels, probs)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── 2b: ROC with CI ───────────────────────────────────────────────────────
ax = axes[0]
ax.plot(fpr_main, tpr_main, color="#4c9be8", lw=2,
        label=f"AUC = {auc_mean:.3f} [{auc_lo:.3f}–{auc_hi:.3f}]")
ax.fill_between(mean_fpr, tpr_lo, tpr_hi, alpha=0.2, color="#4c9be8", label="95% CI")
ax.plot([0,1],[0,1],"k--",lw=1)
ax.scatter([1-opt_spec],[opt_sens], color="#e05c5c", s=80, zorder=5,
           label=f"Youden's J  thr={opt_thresh:.2f}")
# Fixed sensitivity operating points
for target_sens, color in [(0.90,"orange"),(0.95,"green")]:
    idx_90 = np.argmin(np.abs(tpr_main - target_sens))
    ax.scatter([fpr_main[idx_90]], [tpr_main[idx_90]], color=color, s=60, zorder=5,
               label=f"Sens≥{target_sens:.0%}")
ax.set_xlabel("1 − Specificity (FPR)")
ax.set_ylabel("Sensitivity (TPR)")
ax.set_title("ROC Curve with 95% CI", fontsize=12)
ax.legend(fontsize=8)

# ── 2c: Precision-Recall curve ────────────────────────────────────────────
ax = axes[1]
prec, rec, _ = precision_recall_curve(labels, probs)
ap = average_precision_score(labels, probs)
baseline = labels.mean()
ax.plot(rec, prec, color="#6dbf8f", lw=2, label=f"AP = {ap:.3f}")
ax.axhline(baseline, color="grey", lw=1, ls="--", label=f"Baseline (prevalence={baseline:.2f})")
ax.set_xlabel("Recall (Sensitivity)")
ax.set_ylabel("Precision (PPV)")
ax.set_title("Precision-Recall Curve", fontsize=12)
ax.legend(fontsize=9)

# ── 2d: Dual confusion matrix ─────────────────────────────────────────────
ax = axes[2]
cm = metrics["confusion_matrix"]
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
fig.colorbar(im, ax=ax, fraction=0.046)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]}\n({cm_norm[i,j]:.1%})",
                ha="center", va="center", fontsize=11,
                color="white" if cm_norm[i,j]>0.5 else "black")
ax.set_xticks([0,1]); ax.set_xticklabels(["Normal","Fracture"])
ax.set_yticks([0,1]); ax.set_yticklabels(["Normal","Fracture"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix", fontsize=12)

fig.suptitle("Model Performance — Test Set", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "2_performance.png", bbox_inches="tight")
plt.show()

## 3. Per-Region Subgroup Analysis

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

regions      = ["distal", "complete", "proximal"]
region_color = {"distal": "#4c9be8", "complete": "#e05c5c", "proximal": "#6dbf8f"}
region_rows  = []

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── ROC per region ────────────────────────────────────────────────────────
ax = axes[0]
ax.plot([0,1],[0,1],"k--",lw=1)
for region in regions:
    mask  = test_df["region"] == region
    l_r   = labels[mask]
    p_r   = probs[mask]
    if len(np.unique(l_r)) < 2:
        continue
    auc_r  = roc_auc_score(l_r, p_r)
    fpr_r, tpr_r, _ = roc_curve(l_r, p_r)
    pred_r = (p_r >= 0.5).astype(int)
    tn = ((pred_r==0)&(l_r==0)).sum(); fp = ((pred_r==1)&(l_r==0)).sum()
    fn = ((pred_r==0)&(l_r==1)).sum(); tp = ((pred_r==1)&(l_r==1)).sum()
    sens_r = tp/(tp+fn) if (tp+fn)>0 else 0
    spec_r = tn/(tn+fp) if (tn+fp)>0 else 0
    region_rows.append({"Region": region.capitalize(), "N": mask.sum(),
                        "AUC": f"{auc_r:.3f}", "Sensitivity": f"{sens_r:.3f}",
                        "Specificity": f"{spec_r:.3f}"})
    ax.plot(fpr_r, tpr_r, color=region_color[region], lw=2,
            label=f"{region.capitalize()} (AUC={auc_r:.3f}, n={mask.sum()})")

ax.set_xlabel("1 − Specificity")
ax.set_ylabel("Sensitivity")
ax.set_title("ROC by Anatomical Region", fontsize=12)
ax.legend(fontsize=9)

# ── Bar chart: AUC per region ─────────────────────────────────────────────
ax = axes[1]
region_df = pd.DataFrame(region_rows)
aucs = region_df["AUC"].astype(float)
bars = ax.bar(region_df["Region"], aucs,
              color=[region_color[r.lower()] for r in region_df["Region"]],
              edgecolor="white", width=0.5)
for bar, val in zip(bars, aucs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f"{val:.3f}", ha="center", fontsize=11)
ax.set_ylim(0.5, 1.05)
ax.set_ylabel("AUC-ROC")
ax.set_title("AUC per Anatomical Region", fontsize=12)
ax.axhline(0.5, color="grey", lw=1, ls="--", label="Random")

fig.suptitle("Subgroup Analysis by Anatomical Region", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "3_subgroup_analysis.png", bbox_inches="tight")
plt.show()

print("\nSubgroup metrics table:")
print(region_df.to_string(index=False))

## 4. Threshold Analysis — Clinical Operating Points

In [ ]:
fpr_all, tpr_all, thresholds_all = roc_curve(labels, probs)
spec_all = 1 - fpr_all
j_scores = tpr_all - fpr_all   # Youden's J

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Sensitivity & Specificity vs threshold ────────────────────────────────
ax = axes[0]
ax.plot(thresholds_all, tpr_all[:-1] if len(thresholds_all)<len(tpr_all) else tpr_all,
        color=COLORS["fracture"], lw=2, label="Sensitivity")
ax.plot(thresholds_all, spec_all[:-1] if len(thresholds_all)<len(spec_all) else spec_all,
        color=COLORS["normal"],   lw=2, label="Specificity")
ax.axvline(opt_thresh, color="grey", lw=1, ls="--",
           label=f"Youden's J  thr={opt_thresh:.2f}")
ax.axvline(0.5, color="black", lw=1, ls=":", label="Default thr=0.5")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Rate")
ax.set_title("Sensitivity & Specificity vs Threshold", fontsize=12)
ax.legend(fontsize=9)

# ── Youden's J curve ──────────────────────────────────────────────────────
ax = axes[1]
# align arrays
j_plot = tpr_all[1:] - fpr_all[1:]
th_plot = thresholds_all
if len(th_plot) != len(j_plot):
    min_len = min(len(th_plot), len(j_plot))
    th_plot, j_plot = th_plot[:min_len], j_plot[:min_len]
ax.plot(th_plot, j_plot, color="purple", lw=2)
best_j = np.argmax(j_plot)
ax.scatter([th_plot[best_j]], [j_plot[best_j]], color="gold", s=100, zorder=5,
           label=f"Max J={j_plot[best_j]:.3f} @ thr={th_plot[best_j]:.2f}")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Youden's J  (Sensitivity + Specificity − 1)")
ax.set_title("Youden's J Statistic", fontsize=12)
ax.legend(fontsize=9)

fig.suptitle("Threshold Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "4_threshold_analysis.png", bbox_inches="tight")
plt.show()

# ── Operating points table ────────────────────────────────────────────────
def metrics_at_threshold(labels, probs, thr):
    pred = (probs >= thr).astype(int)
    tn = ((pred==0)&(labels==0)).sum(); fp = ((pred==1)&(labels==0)).sum()
    fn = ((pred==0)&(labels==1)).sum(); tp = ((pred==1)&(labels==1)).sum()
    return {
        "Sensitivity": tp/(tp+fn) if tp+fn>0 else 0,
        "Specificity": tn/(tn+fp) if tn+fp>0 else 0,
        "PPV": tp/(tp+fp) if tp+fp>0 else 0,
        "NPV": tn/(tn+fn) if tn+fn>0 else 0,
        "Accuracy": (tp+tn)/len(labels),
    }

# Threshold for sensitivity >= 0.90
idx_90 = np.argmin(np.abs(tpr_all - 0.90))
thr_90 = thresholds_all[min(idx_90, len(thresholds_all)-1)]
idx_95 = np.argmin(np.abs(tpr_all - 0.95))
thr_95 = thresholds_all[min(idx_95, len(thresholds_all)-1)]

op_rows = []
for name, thr in [("Default (0.5)", 0.5), ("Youden's J", opt_thresh),
                   ("Sens ≥ 90%", thr_90), ("Sens ≥ 95%", thr_95)]:
    m = metrics_at_threshold(labels, probs, thr)
    op_rows.append({"Operating Point": name, "Threshold": f"{thr:.3f}",
                    **{k: f"{v:.3f}" for k, v in m.items()}})

print("\nClinical Operating Points:")
print(pd.DataFrame(op_rows).to_string(index=False))

## 5. Calibration Analysis

In [ ]:
from sklearn.calibration import calibration_curve

N_BINS = 10
frac_pos, mean_pred = calibration_curve(labels, probs, n_bins=N_BINS, strategy="uniform")

# ECE = weighted average of |bin_confidence - bin_accuracy|
bin_edges = np.linspace(0, 1, N_BINS + 1)
bin_counts = np.histogram(probs, bins=bin_edges)[0]
ece = np.sum(bin_counts * np.abs(mean_pred - frac_pos)) / len(probs)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot([0,1],[0,1], "k--", lw=1, label="Perfect calibration")
ax.plot(mean_pred, frac_pos, "o-", color="#7b68ee", lw=2, ms=7,
        label=f"Model  (ECE={ece:.3f})")
ax.fill_between(mean_pred, mean_pred, frac_pos, alpha=0.15, color="#e05c5c")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.set_title("Calibration Curve (Reliability Diagram)", fontsize=12)
ax.legend(fontsize=10)

ax = axes[1]
ax.hist(probs[labels==1], bins=20, alpha=0.6, color=COLORS["fracture"], label="Fracture")
ax.hist(probs[labels==0], bins=20, alpha=0.6, color=COLORS["normal"],   label="Normal")
ax.axvline(0.5,        color="grey",  lw=1, ls=":",  label="Thr=0.5")
ax.axvline(opt_thresh, color="black", lw=1, ls="--", label=f"Youden's J={opt_thresh:.2f}")
ax.set_xlabel("Predicted Fracture Probability")
ax.set_ylabel("Count")
ax.set_title("Probability Distribution by Class", fontsize=12)
ax.legend(fontsize=9)

fig.suptitle(f"Calibration Analysis  |  ECE = {ece:.3f}", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "5_calibration.png", bbox_inches="tight")
plt.show()
print(f"Expected Calibration Error (ECE): {ece:.4f}  (lower is better; 0 = perfect)")

## 6. Feature Space Visualisation (t-SNE & UMAP)

In [ ]:
from sklearn.manifold import TSNE

# ── Extract penultimate features via forward hook ─────────────────────────
features_list = []

def _hook(module, inp, out):
    # backbone outputs pooled features (B, num_features)
    features_list.append(out.detach().cpu())

hook_handle = model.backbone.register_forward_hook(_hook)

model.eval()
all_labels_full, all_probs_full = [], []

with torch.no_grad():
    for imgs, lbls in dls["test"]:
        _ = model(imgs.to(DEVICE))
        all_labels_full.extend(lbls.numpy())
        all_probs_full.extend(torch.softmax(model(imgs.to(DEVICE)), 1)[:,1].cpu().numpy())

hook_handle.remove()
features = torch.cat(features_list, dim=0).numpy()
feat_labels  = np.array(all_labels_full[:len(features)])
feat_regions = test_df["region"].values[:len(features)]
feat_correct = test_df["correct"].values[:len(features)]

print(f"Feature matrix: {features.shape}")
print("Running t-SNE...")
tsne  = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
emb_tsne = tsne.fit_transform(features)
print("t-SNE done.")

In [ ]:
try:
    import umap
    print("Running UMAP...")
    reducer  = umap.UMAP(n_components=2, random_state=42)
    emb_umap = reducer.fit_transform(features)
    print("UMAP done.")
    has_umap = True
except ImportError:
    print("umap-learn not installed — skipping UMAP (pip install umap-learn)")
    has_umap = False

In [ ]:
n_cols = 4 if has_umap else 3
fig, axes = plt.subplots(1, n_cols, figsize=(6*n_cols, 5))

def scatter_emb(ax, emb, color_arr, cmap_or_colors, title, labels_legend=None):
    if isinstance(cmap_or_colors, dict):
        for key, col in cmap_or_colors.items():
            mask = color_arr == key
            ax.scatter(emb[mask,0], emb[mask,1], c=col, s=12, alpha=0.7, label=str(key))
    else:
        sc = ax.scatter(emb[:,0], emb[:,1], c=color_arr, cmap=cmap_or_colors,
                        s=12, alpha=0.7)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    if isinstance(cmap_or_colors, dict):
        ax.legend(markerscale=2, fontsize=8)

label_colors  = {0: COLORS["normal"], 1: COLORS["fracture"]}
label_arr     = np.where(feat_labels==1, "Fracture", "Normal")
label_colors2 = {"Fracture": COLORS["fracture"], "Normal": COLORS["normal"]}
region_cols   = {"distal": "#4c9be8", "complete": "#e05c5c", "proximal": "#6dbf8f"}
correct_cols  = {1: COLORS["correct"], 0: COLORS["wrong"]}
correct_arr   = np.where(feat_correct==1, "Correct", "Wrong")
correct_cols2 = {"Correct": COLORS["correct"], "Wrong": COLORS["wrong"]}

scatter_emb(axes[0], emb_tsne, label_arr,    label_colors2,  "t-SNE: Fracture vs Normal")
scatter_emb(axes[1], emb_tsne, feat_regions, region_cols,    "t-SNE: Anatomical Region")
scatter_emb(axes[2], emb_tsne, correct_arr,  correct_cols2,  "t-SNE: Correct vs Wrong")
if has_umap:
    scatter_emb(axes[3], emb_umap, label_arr, label_colors2, "UMAP: Fracture vs Normal")

fig.suptitle("Feature Space Visualisation (Penultimate Layer)", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "6_feature_space.png", bbox_inches="tight")
plt.show()

## 7. Grad-CAM — Model Interpretability

In [ ]:
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image
    HAS_GRADCAM = True
except ImportError:
    print("grad-cam not installed — pip install grad-cam")
    HAS_GRADCAM = False

if HAS_GRADCAM:
    # Target: last convolutional block of backbone
    if hasattr(model.backbone, "blocks"):
        target_layers = [model.backbone.blocks[-1]]
    else:
        # fallback: last child module
        target_layers = [list(model.backbone.children())[-1]]

    cam = GradCAM(model=model, target_layers=target_layers)
    print(f"GradCAM target layer: {target_layers[0].__class__.__name__}")

    def get_gradcam_overlay(img_path, fracture_target=True):
        """
        Load an image, run Grad-CAM, return (original_rgb, heatmap_overlay).
        """
        transform = get_val_transforms()
        img_pil   = Image.open(img_path).convert("L")
        inp_tensor = transform(img_pil).unsqueeze(0).to(DEVICE)

        targets = [ClassifierOutputTarget(1 if fracture_target else 0)]
        grayscale_cam = cam(input_tensor=inp_tensor, targets=targets)[0]   # (H, W)

        # Original image as float RGB (0-1) for overlay
        img_arr = np.array(img_pil.resize((224, 224), Image.BILINEAR), dtype=np.float32) / 255.0
        img_rgb = np.stack([img_arr]*3, axis=-1)  # (H, W, 3)

        overlay = show_cam_on_image(img_rgb, grayscale_cam, use_rgb=True)
        return img_rgb, overlay, grayscale_cam

    print("Grad-CAM ready.")

In [ ]:
if HAS_GRADCAM:
    N_EXAMPLES = 4  # examples per outcome group

    groups = {
        "True Positive\n(fracture correctly detected)": test_df[(test_df["label_id"]==1) & (test_df["pred"]==1)],
        "True Negative\n(normal correctly identified)": test_df[(test_df["label_id"]==0) & (test_df["pred"]==0)],
        "False Positive\n(false alarm — normal called fracture)": test_df[(test_df["label_id"]==0) & (test_df["pred"]==1)],
        "False Negative\n(missed fracture)": test_df[(test_df["label_id"]==1) & (test_df["pred"]==0)],
    }

    for group_name, group_df in groups.items():
        sample = group_df.sample(min(N_EXAMPLES, len(group_df)), random_state=42)
        if len(sample) == 0:
            print(f"No examples for: {group_name.splitlines()[0]}")
            continue

        n = len(sample)
        fig, axes = plt.subplots(n, 3, figsize=(9, 3.2*n))
        if n == 1:
            axes = axes[np.newaxis, :]

        fig.suptitle(group_name, fontsize=13, fontweight="bold", y=1.01)

        for row_i, (_, row) in enumerate(sample.iterrows()):
            orig, overlay, heatmap = get_gradcam_overlay(row["path"], fracture_target=True)
            prob_val = row["prob"]

            axes[row_i, 0].imshow(orig, cmap="gray")
            axes[row_i, 0].set_title(f"Original\n{row['case_id']} | {row['region']}", fontsize=8)

            axes[row_i, 1].imshow(heatmap)
            axes[row_i, 1].set_title(f"Grad-CAM  (P(frac)={prob_val:.2f})", fontsize=8)

            axes[row_i, 2].imshow(overlay)
            axes[row_i, 2].set_title("Overlay", fontsize=8)

            for ax in axes[row_i]:
                ax.axis("off")

        plt.tight_layout()
        safe_name = group_name.splitlines()[0].lower().replace(" ","_").replace("(","").replace(")","")
        fig.savefig(FIG_DIR / f"7_gradcam_{safe_name}.png", bbox_inches="tight")
        plt.show()

## 8. Confidence & Error Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Confidence histogram: correct vs wrong ────────────────────────────────
ax = axes[0]
correct_mask = test_df["correct"] == 1
ax.hist(probs[correct_mask],  bins=20, alpha=0.65, color=COLORS["correct"], label="Correct")
ax.hist(probs[~correct_mask], bins=20, alpha=0.65, color=COLORS["wrong"],   label="Wrong")
ax.axvline(0.5,        color="grey",  lw=1, ls=":")
ax.axvline(opt_thresh, color="black", lw=1, ls="--", label=f"Youden's J={opt_thresh:.2f}")
ax.set_xlabel("Predicted Fracture Probability")
ax.set_ylabel("Count")
ax.set_title("Confidence: Correct vs Wrong Predictions", fontsize=12)
ax.legend()

# ── Violin plot: probability by outcome group (TP/TN/FP/FN) ──────────────
ax = axes[1]
outcome_labels, outcome_probs = [], []
label_map = {(1,1):"TP", (0,0):"TN", (0,1):"FP", (1,0):"FN"}
for (true_l, pred_l), name in label_map.items():
    mask = (test_df["label_id"]==true_l) & (test_df["pred"]==pred_l)
    outcome_probs.append(probs[mask])
    outcome_labels.append(f"{name}\n(n={mask.sum()})")

vp = ax.violinplot(outcome_probs, showmedians=True, showextrema=True)
colors_violin = [COLORS["correct"],COLORS["correct"],COLORS["wrong"],COLORS["wrong"]]
for body, col in zip(vp["bodies"], colors_violin):
    body.set_facecolor(col); body.set_alpha(0.6)
ax.set_xticks(range(1, len(outcome_labels)+1))
ax.set_xticklabels(outcome_labels, fontsize=9)
ax.set_ylabel("Predicted Fracture Probability")
ax.set_title("Confidence Distribution by Outcome", fontsize=12)
ax.axhline(0.5, color="grey", lw=1, ls=":")

fig.suptitle("Confidence & Error Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "8_confidence_analysis.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Most confident mistakes ────────────────────────────────────────────────
for mistake_type, label_val, pred_val, title in [
    ("fp", 0, 1, "Most Confident False Positives (Normal → Predicted Fracture)"),
    ("fn", 1, 0, "Most Confident False Negatives (Fracture → Predicted Normal)"),
]:
    mask   = (test_df["label_id"]==label_val) & (test_df["pred"]==pred_val)
    errors = test_df[mask].copy()
    # Most confident: FP → highest fracture prob; FN → lowest fracture prob
    errors = errors.sort_values("prob", ascending=(mistake_type=="fn")).head(8)

    n = len(errors)
    if n == 0:
        print(f"No {mistake_type.upper()} examples in test set.")
        continue

    cols = min(n, 4)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, errors.iterrows()):
        img = Image.open(row["path"]).convert("L")
        ax.imshow(img, cmap="gray")
        ax.set_title(f"{row['case_id']}\nP(frac)={row['prob']:.2f} | {row['region']}",
                     fontsize=8)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    fig.savefig(FIG_DIR / f"8_mistakes_{mistake_type}.png", bbox_inches="tight")
    plt.show()

## 9. Publication-Ready Summary Table

In [ ]:
from sklearn.metrics import f1_score as f1

# ── Final metrics with CI ──────────────────────────────────────────────────
summary_rows = [
    ("AUC-ROC",         *ci["auc"]),
    ("Sensitivity",     *ci["sens"]),
    ("Specificity",     *ci["spec"]),
    ("Accuracy",        *ci["acc"]),
    ("PPV (Precision)", *ci["ppv"]),
    ("NPV",             *ci["npv"]),
    ("F1-Score",        *ci["f1"]),
    ("Avg. Precision",  *ci["ap"]),
]

summary_df = pd.DataFrame(
    [(name, f"{m:.3f}", f"[{lo:.3f}, {hi:.3f}]")
     for name, m, lo, hi in summary_rows],
    columns=["Metric", "Value", "95% CI"]
)
# ECE has no CI
summary_df = pd.concat([
    summary_df,
    pd.DataFrame([["ECE", f"{ece:.3f}", "—"]], columns=["Metric","Value","95% CI"]),
    pd.DataFrame([["Optimal Threshold", f"{opt_thresh:.3f}", "—"]], columns=["Metric","Value","95% CI"]),
], ignore_index=True)

print("=" * 50)
print("PUBLICATION SUMMARY TABLE")
print("=" * 50)
print(summary_df.to_string(index=False))

In [ ]:
# ── LaTeX table (copy-paste into thesis) ──────────────────────────────────
latex = summary_df.to_latex(
    index=False,
    caption="Performance metrics of the fracture classification model on the held-out test set. "
            "95\\% confidence intervals estimated via stratified bootstrap (1000 iterations).",
    label="tab:model_performance",
    escape=True,
    column_format="lcc",
)
print("\n── LaTeX table ─────────────────────────────────────────────")
print(latex)

# ── Save all outputs ─────────────────────────────────────────────────────
summary_df.to_csv(FIG_DIR / "9_summary_metrics.csv", index=False)
with open(FIG_DIR / "9_summary_metrics.tex", "w") as f:
    f.write(latex)

print(f"\nAll thesis figures saved to: {FIG_DIR}")
print("Files generated:")
for p in sorted(FIG_DIR.iterdir()):
    print(f"  {p.name}")